# Interactive Chat Test (py/ipynb 대화형 예시)

이 노트북은 Copyjoe API를 대화형으로 테스트합니다.
- `/api/v1/copy/generate`
- `/api/v1/web/landing/analyze`
- `/health`

아래 `chat_turn(...)` 함수에 명령을 넣어 대화형으로 확인할 수 있습니다.

In [ ]:
import json
import os

import requests

BASE_URL = os.getenv("COPYJOE_BASE_URL", "http://127.0.0.1:8000")
session = requests.Session()

payload = {
    "product_name": "Copyjoe",
    "target_audience": "퍼포먼스 마케터",
    "pain_point": "카피 생성 속도가 느리다",
    "differentiator": "RAG + Tavily 기반 근거 중심 생성",
    "tone": "신뢰형",
    "objective": "click",
    "styles": ["head", "body", "cta", "slogan", "sns", "description"],
    "channel": "상세페이지",
    "language": "ko",
    "web_search_mode": False,
    "use_rag": True,
    "top_k": 5,
}

turns = []


In [ ]:
def chat_turn(command: str):
    command = command.strip()
    turns.append(("user", command))

    if command.startswith("set pain:"):
        payload["pain_point"] = command.split(":", 1)[1].strip()
        reply = f"pain_point updated: {payload['pain_point']}"
    elif command.startswith("set objective:"):
        payload["objective"] = command.split(":", 1)[1].strip()
        reply = f"objective updated: {payload['objective']}"
    elif command == "toggle web":
        payload["web_search_mode"] = not payload["web_search_mode"]
        reply = f"web_search_mode={payload['web_search_mode']}"
    elif command == "toggle rag":
        payload["use_rag"] = not payload["use_rag"]
        reply = f"use_rag={payload['use_rag']}"
    elif command == "generate":
        res = session.post(f"{BASE_URL}/api/v1/copy/generate", json=payload, timeout=180)
        res.raise_for_status()
        body = res.json()
        reply = json.dumps({
            "head": body.get("head"),
            "cta": body.get("cta"),
            "sources": len(body.get("sources", [])),
            "rationale": body.get("rationale"),
        }, ensure_ascii=False, indent=2)
    elif command.startswith("landing url:"):
        url = command.split(":", 1)[1].strip()
        res = session.post(f"{BASE_URL}/api/v1/web/landing/analyze", json={"url": url}, timeout=180)
        res.raise_for_status()
        body = res.json()
        reply = json.dumps({
            "url": body.get("url"),
            "h1": body.get("h1", []),
            "h2_count": len(body.get("h2", [])),
            "cta_count": len(body.get("cta_buttons", [])),
            "body_preview": body.get("body", "")[:220],
        }, ensure_ascii=False, indent=2)
    elif command.startswith("landing query:"):
        query = command.split(":", 1)[1].strip()
        res = session.post(f"{BASE_URL}/api/v1/web/landing/analyze", json={"query": query, "max_results": 3}, timeout=180)
        res.raise_for_status()
        body = res.json()
        reply = json.dumps({
            "url": body.get("url"),
            "from_tavily": body.get("from_tavily"),
            "h1": body.get("h1", []),
            "h2_count": len(body.get("h2", [])),
            "cta_count": len(body.get("cta_buttons", [])),
        }, ensure_ascii=False, indent=2)
    elif command == "health":
        res = session.get(f"{BASE_URL}/health", timeout=20)
        res.raise_for_status()
        reply = json.dumps(res.json(), ensure_ascii=False, indent=2)
    else:
        reply = "unknown command"

    turns.append(("assistant", reply))
    print("assistant:")
    print(reply)
    return reply


In [ ]:
chat_turn("health")
chat_turn("set pain: 랜딩 전환률이 낮다")
chat_turn("generate")
chat_turn("landing url: https://example.com")


In [ ]:
assert len(turns) >= 8, "대화 턴이 충분하지 않습니다."
assert turns[0][0] == "user"
assert turns[1][0] == "assistant"
print("interactive chat test: PASS")
